# DSRL offline-to-online 실행 노트북 v3 (Colab Pro+)

v2와 다른 점
- 코드를 포크(`msp0617/dsrl`, 브랜치 `o2o`)에서 받는다. 노트북에서 sed로 덧칠하던 패치가 코드에 들어갔다.
- 체크포인트/resume, CSV 로깅, 환경 스텝 단위 예산이 들어갔다. 세션이 끊기면 같은 명령을 다시 실행하면 이어진다.
- site-packages 패치는 `colab/patch_env.py` 한 번으로 끝난다.

설치 셀(1~8)은 v2에서 실제로 성공한 순서를 그대로 유지했다.
**세션 정책**: 코드/패키지는 매 세션 새로 설치, Drive에는 체크포인트·로그·config만 보관.

## 0. Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJ = '/content/drive/MyDrive/dsrl_project'
for d in ['ckpt', 'logs', 'cfg_backup', 'dppo_log']:
    os.makedirs(f'{PROJ}/{d}', exist_ok=True)
print('project dir:', PROJ)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. condacolab
아래 실행하면 **커널이 자동 재시작**됩니다. 정상이니 재시작 후 2번부터 이어서 실행하세요.
(이미 설치했다면 건너뛰기)

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()   # 여기서 커널 재시작

## 2. 재시작 후: Drive 다시 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJ = '/content/drive/MyDrive/dsrl_project'
print('ok')

## 3. 저장소 클론
`.gitmodules`가 ssh 주소라서 그냥 클론하면 서브모듈(dppo, stable-baselines3)이 빈 폴더가 됨.
아래 `insteadOf` 설정이 그걸 막아줌.

In [ ]:
%%bash
git config --global url."https://github.com/".insteadOf "git@github.com:"

cd /content
rm -rf dsrl
git clone --recurse-submodules -b o2o https://github.com/msp0617/dsrl.git
cd dsrl

git log --oneline -1
echo "=== dppo ==="
ls dppo | head -3
echo "=== stable-baselines3 ==="
ls stable-baselines3 | head -3

두 폴더에 파일이 보여야 함. 비어 있으면 아래 셀로 서브모듈만 다시 받기.

In [ ]:
%%bash
cd /content/dsrl
git submodule sync --recursive
git submodule update --init --recursive
ls dppo | head

## 4. conda 환경 (Python 3.10)

In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh
conda env remove -n dsrl -y 2>/dev/null
conda create -n dsrl python=3.10 -y -q
echo created

## 5. 설치
5~15분 걸립니다. `-q`를 뺐으니 진행 상황이 보임.

In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh
conda activate dsrl
cd /content/dsrl/dppo
pip install -e .
pip install -e ".[robomimic]"
echo "=== dppo done ==="


In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh
conda activate dsrl
cd /content/dsrl/stable-baselines3
pip install -e .
pip install gdown
echo "=== sb3 done ==="


### 검증

In [ ]:
%%bash
set -e
source /usr/local/etc/profile.d/conda.sh
conda activate dsrl

python -m pip install "mujoco==3.1.6"

python -c "import mujoco; print('mujoco', mujoco.__version__)"

In [ ]:
%%bash
set -e
source /usr/local/etc/profile.d/conda.sh
conda activate dsrl

# Robomimic/Robosuite 기본 준비
python -m pip install "cython<3" patchelf

# DPPO가 지정한 조합으로 직접 설치
python -m pip install \
  "robosuite @ git+https://github.com/ARISE-Initiative/robosuite.git@v1.4.1"

python -m pip install \
  "robomimic @ git+https://github.com/ARISE-Initiative/robomimic.git"

echo "=== robomimic + robosuite installed ==="

In [ ]:
%%bash
set -e
source /usr/local/etc/profile.d/conda.sh
conda activate dsrl

echo "=== 1. 잘못 올라간 NumPy/OpenCV 복구 ==="
python -m pip install --force-reinstall \
  "numpy==1.26.4" \
  "opencv-python==4.9.0.80"

echo "=== 2. DPPO의 Torch 2.4와 맞는 torchvision 고정 ==="
python -m pip install \
  "torch==2.4.0" \
  "torchvision==0.19.0"

echo "=== 3. egl_probe 빌드용 구버전 CMake ==="
python -m pip install "cmake==3.31.6"
cmake --version

echo "=== 4. egl_probe 설치 ==="
python -m pip install --no-build-isolation "egl_probe==1.0.2"

echo "=== 5. robomimic 호환 버전 설치 ==="
python -m pip install "robomimic==0.3.0"

echo "=== recovery completed ==="

In [ ]:
%%bash
set -e
source /usr/local/etc/profile.d/conda.sh
conda activate dsrl

export MUJOCO_GL=egl
export PYOPENGL_PLATFORM=egl

python - <<'PY'
import sys
print("python     ", sys.executable)

import numpy
print("numpy      ", numpy.__version__)

import torch
print("torch      ", torch.__version__, "| cuda:", torch.cuda.is_available())

import torchvision
print("torchvision", torchvision.__version__)

import mujoco
print("mujoco     ", mujoco.__version__)

import robomimic
print("robomimic  ", robomimic.__version__)

import robosuite
print("robosuite  ", robosuite.__version__)

import stable_baselines3 as sb3
print("sb3        ", sb3.__version__)
PY

In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh
conda activate dsrl
python /usr/local/envs/dsrl/lib/python3.10/site-packages/robosuite/scripts/setup_macros.py

## 6. π_dp 체크포인트
README 링크의 Drive 폴더를 `dppo/log`에 배치. 첫 세션만 다운로드하고 Drive에 복사해두면 다음부터는 복사만.

In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh
conda activate dsrl
PROJ=/content/drive/MyDrive/dsrl_project
mkdir -p /content/dsrl/dppo/log

if [ -z "$(ls -A $PROJ/dppo_log 2>/dev/null)" ]; then
  echo "--- 첫 다운로드 ---"
  cd /content/dsrl/dppo/log
  gdown --folder https://drive.google.com/drive/folders/1kzC49RRFOE7aTnJh_7OvJ1K5XaDmtuh1
  cp -r /content/dsrl/dppo/log/. $PROJ/dppo_log/
else
  echo "--- Drive에서 복사 ---"
  cp -r $PROJ/dppo_log/. /content/dsrl/dppo/log/
fi
find /content/dsrl/dppo/log -maxdepth 3 | head -40

In [ ]:
%%bash
cd /content/dsrl

for FILE in \
  dppo/log/robomimic-pretrain/can/can_pre_diffusion_mlp_ta4_td20/2024-06-28_13-29-54/checkpoint/state_5000.pt \
  dppo/log/robomimic/can/normalization.npz
do
  test -f "$FILE" && echo "OK: $FILE" || echo "MISSING: $FILE"
done

In [ ]:
%%bash
set -e

RUNTIME=/content/dsrl/dppo/log
DRIVE=/content/drive/MyDrive/dsrl_project/dppo_log

CKPT_REL=robomimic-pretrain/can/can_pre_diffusion_mlp_ta4_td20/2024-06-28_13-29-54/checkpoint/state_5000.pt
NORM_REL=robomimic/can/normalization.npz

mkdir -p "$RUNTIME" "$DRIVE"

CKPT_SRC=$(find "$RUNTIME" "$DRIVE" \
  -type f -path "*/$CKPT_REL" -print -quit 2>/dev/null || true)

NORM_SRC=$(find "$RUNTIME" "$DRIVE" \
  -type f -path "*/$NORM_REL" -print -quit 2>/dev/null || true)

echo "checkpoint: ${CKPT_SRC:-NOT_FOUND}"
echo "normalization: ${NORM_SRC:-NOT_FOUND}"

if [[ -z "$CKPT_SRC" || -z "$NORM_SRC" ]]; then
  echo "기존 다운로드에서 파일을 찾지 못했습니다."
  exit 2
fi

CKPT_DST="$RUNTIME/$CKPT_REL"
NORM_DST="$RUNTIME/$NORM_REL"

mkdir -p "$(dirname "$CKPT_DST")" "$(dirname "$NORM_DST")"

[[ "$CKPT_SRC" == "$CKPT_DST" ]] || cp -f "$CKPT_SRC" "$CKPT_DST"
[[ "$NORM_SRC" == "$NORM_DST" ]] || cp -f "$NORM_SRC" "$NORM_DST"

# 다음 세션을 위해 Drive에도 정확한 구조로 저장
mkdir -p "$DRIVE/$(dirname "$CKPT_REL")"
mkdir -p "$DRIVE/$(dirname "$NORM_REL")"
cp -f "$CKPT_DST" "$DRIVE/$CKPT_REL"
cp -f "$NORM_DST" "$DRIVE/$NORM_REL"

echo "=== READY ==="
ls -lh "$CKPT_DST" "$NORM_DST"

## 7. 환경 변수
headless 렌더링과 wandb 비활성 설정.

In [ ]:
%%bash
cat > /content/env.sh <<'EOS'
export MUJOCO_GL=egl
export PYOPENGL_PLATFORM=egl
export WANDB_MODE=disabled
EOS

cat /content/env.sh

## 8. Config 확인
Can config의 키 이름을 보고 smoke test용 override를 정합니다 (총 step 수, 저장 주기 등).

In [ ]:
!cat /content/dsrl/cfg/robomimic/dsrl_can.yaml

## 9. site-packages 패치
robomimic이 mujoco_py를 무조건 import하는 문제를 고친다. **세션마다 한 번** 실행.

In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh
conda activate dsrl
python /content/dsrl/colab/patch_env.py

## 10. 스모크 테스트 + resume 확인
작은 설정으로 두 번 나눠 돌린다. 두 번째 실행이 `[resume]` 을 찍고 이어가면 정상.

In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh && conda activate dsrl
source /content/env.sh
cd /content/dsrl
export HYDRA_FULL_ERROR=1

python train_dsrl.py --config-path=cfg/robomimic --config-name=dsrl_can.yaml \
  exp_id=smoke_resume \
  log_dir=/content/drive/MyDrive/dsrl_project/logs \
  env.n_envs=1 env.n_eval_envs=1 num_evals=1 \
  eval_schedule.every_env_early=400 eval_schedule.early_until_env=100000 \
  ckpt_every_env_steps=400 \
  train.init_rollout_steps=50 train.utd=1 train.noise_critic_grad_steps=1 \
  train.batch_size=32 train.layer_size=256 train.num_layers=2 \
  train.buffer_size=20000 train.total_env_steps=1200

In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh && conda activate dsrl
source /content/env.sh
cd /content/dsrl
export HYDRA_FULL_ERROR=1

# 같은 명령에 목표만 늘린다. 앞 실행의 체크포인트에서 이어져야 한다.
python train_dsrl.py --config-path=cfg/robomimic --config-name=dsrl_can.yaml \
  exp_id=smoke_resume \
  log_dir=/content/drive/MyDrive/dsrl_project/logs \
  env.n_envs=1 env.n_eval_envs=1 num_evals=1 \
  eval_schedule.every_env_early=400 eval_schedule.early_until_env=100000 \
  ckpt_every_env_steps=400 \
  train.init_rollout_steps=50 train.utd=1 train.noise_critic_grad_steps=1 \
  train.batch_size=32 train.layer_size=256 train.num_layers=2 \
  train.buffer_size=20000 train.total_env_steps=2000 2>&1 | grep -E "resume|budget|eval\]|ckpt\]" | head -20

In [ ]:
PROJ = '/content/drive/MyDrive/dsrl_project'
!ls -lh {PROJ}/logs/smoke_resume {PROJ}/logs/smoke_resume/checkpoint
!cat {PROJ}/logs/smoke_resume/checkpoint/run_state.json
!head -3 {PROJ}/logs/smoke_resume/eval_log.csv

## 11. 처리량 측정
**본 실험 하이퍼파라미터 그대로**, 초기 rollout만 줄여서 학습 구간 속도를 잰다.
결과의 `env steps/s` 로 300k 한 run에 몇 시간 걸리는지 나온다.

In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh && conda activate dsrl
source /content/env.sh
cd /content/dsrl
export HYDRA_FULL_ERROR=1

# init rollout 200스텝(=3200 env step) + 학습 10000 env step
time python train_dsrl.py --config-path=cfg/robomimic --config-name=dsrl_can.yaml \
  exp_id=tput_can seed=0 resume=False \
  log_dir=/content/drive/MyDrive/dsrl_project/logs \
  train.init_rollout_steps=200 train.total_env_steps=13200 \
  eval_schedule.every_env_early=5000 \
  ckpt_every_env_steps=5000 save_replay_buffer=False

In [ ]:
!source /usr/local/etc/profile.d/conda.sh && conda activate dsrl && \
 python /content/dsrl/colab/throughput.py \
   /content/drive/MyDrive/dsrl_project/logs/tput_can --target 300000

## 12. 본 실험
한 세션에 한 run. `VARIANT`/`SEED`만 바꿔서 세션을 나눠 띄운다.
백그라운드로 돌리므로 셀은 바로 끝나고, 13번으로 진행을 본다.
세션이 죽으면 **이 셀을 그대로 다시 실행**하면 이어진다.

In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh && conda activate dsrl
source /content/env.sh
cd /content/dsrl

VARIANT=baseline      # baseline | warmup | iql
SEED=1
PROJ=/content/drive/MyDrive/dsrl_project
EXP=can_${VARIANT}_s${SEED}

nohup python train_dsrl.py --config-path=cfg/robomimic --config-name=dsrl_can.yaml \
  exp_id=$EXP seed=$SEED variant=$VARIANT \
  log_dir=$PROJ/logs \
  > $PROJ/logs/${EXP}.out 2>&1 &

echo "started $EXP (pid $!) -> $PROJ/logs/${EXP}.out"

## 13. 진행 확인

In [ ]:
EXP = 'can_baseline_s1'
PROJ = '/content/drive/MyDrive/dsrl_project'

!tail -n 3 {PROJ}/logs/{EXP}.out
!echo '--- eval ---' && tail -n 5 {PROJ}/logs/{EXP}/eval_log.csv
!echo '--- ckpt ---' && cat {PROJ}/logs/{EXP}/checkpoint/run_state.json
!source /usr/local/etc/profile.d/conda.sh && conda activate dsrl && \
 python /content/dsrl/colab/throughput.py {PROJ}/logs/{EXP} --target 300000

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

EXP = 'can_baseline_s1'
PROJ = '/content/drive/MyDrive/dsrl_project'
df = pd.read_csv(f'{PROJ}/logs/{EXP}/eval_log.csv')
df = df[df.deterministic == 0]
plt.figure(figsize=(6, 3.5))
plt.plot(df.env_steps, df.success_rate, marker='o', ms=3)
plt.xlabel('environment steps'); plt.ylabel('success rate'); plt.title(EXP)
plt.grid(alpha=.3); plt.show()

## 14. 세션 종료 전 백업

In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh
conda activate dsrl
PROJ=/content/drive/MyDrive/dsrl_project
cp -r /content/dsrl/cfg $PROJ/cfg_backup/
pip freeze > $PROJ/requirements_lock.txt
ls -la $PROJ
echo done